# Steer the Flow with a Directional Tensor

By default the flow spreads equally in every direction, so the shape of a region is distorted the same way along and across the direction in which it has to grow. A `DirectionalTensor` sets a preferred direction for the flow: the velocity is amplified along it (`Dpar`) and, optionally, damped across it (`Dperp`). Only the ratio of the two matters. This lets you decide along which direction a map is stretched, for example to keep the east-west extent of a country and absorb the change in the north-south direction.

This guide shows the effect of a uniform direction and of directions that depend on the position (radial and tangential). The boundary modulators, the `Dx` and `Dy` options and the other velocity modulators are described in [Controlling Geographical Distortions](reduce-boundary-distortion.ipynb).

## Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import carto_flow.data as examples
import carto_flow.flow_cartogram as flow

## Load Geographic Data

The contiguous US states with their population.

In [ ]:
gdf = examples.load_us_census(population=True)

## The Directions

`DirectionalTensor(theta, Dpar, Dperp)` takes the preferred direction `theta` in radians (0 is the positive x-axis, π/2 the positive y-axis). The class methods `DirectionalTensor.radial` and `DirectionalTensor.tangential` set the direction away from a center and around it. Here the center is the centroid of the map. `Dpar=3` and `Dperp=0.5` give a ratio of 6.

In [ ]:
center = gdf.geometry.union_all().centroid.coords[0]

tensors = {
    "Isotropic": None,
    "Horizontal": flow.DirectionalTensor(theta=0.0, Dpar=3.0, Dperp=0.5),
    "Vertical": flow.DirectionalTensor(theta=np.pi / 2, Dpar=3.0, Dperp=0.5),
    "Radial": flow.DirectionalTensor.radial(center=center, Dpar=3.0, Dperp=0.5),
    "Tangential": flow.DirectionalTensor.tangential(center=center, Dpar=3.0, Dperp=0.5),
}

## Compute the Cartograms

The direction is passed to the morph with the `anisotropy` option. The other options are the same for every run, so the cartograms differ only in the direction of the flow.

In [ ]:
options = flow.MorphOptions(grid=256, area_scale=1e-6, show_progress=False)

cartograms = {
    name: flow.create_flow_cartogram(gdf, "Population", options=options.copy_with(anisotropy=tensor))
    for name, tensor in tensors.items()
}

As with any morph, check that every run converged. A strong ratio converges to a larger error than the isotropic flow.

In [ ]:
for name, cartogram in cartograms.items():
    errors = cartogram.get_errors()
    print(
        f"{name:<11} {cartogram.status.name:<10} mean error {errors.mean_error_pct:5.2f} %  max error {errors.max_error_pct:5.2f} %"
    )

## A Uniform Direction

The top row previews the modulator: a uniform probe field that points to the north-east (45°) is passed through it, and the arrows show where the modulator turns it. The probe is turned toward the preferred direction, so the arrows show the preferred direction as a tilt (the isotropic flow leaves the probe as it is). The bottom row shows the cartogram. With a horizontal direction the flow moves the borders along the east-west axis, so regions change their width and keep more of their height; with a vertical direction it is the other way around.

In [ ]:
def compare(names, figsize=(12, 7)):
    fig, axes = plt.subplots(2, len(names), figsize=figsize, squeeze=False)
    for column, name in enumerate(names):
        flow.preview_velocity_modulator(
            tensors[name], gdf, input_angle=np.pi / 4, skip=8, legend=False, ax=axes[0, column]
        )
        cartograms[name].plot(facecolor="Population", legend=False, ax=axes[1, column])
        axes[1, column].set_title("")
        axes[0, column].set_title(name)
        axes[0, column].axis("off")
        axes[1, column].axis("off")
    return fig


_ = compare(["Isotropic", "Horizontal", "Vertical"])

## A Direction That Depends on the Position

A radial direction pushes the flow away from the center, so the map expands or contracts along the rays from the center. A tangential direction moves the borders around the center instead.

In [ ]:
_ = compare(["Isotropic", "Radial", "Tangential"])

## Choosing the Ratio

- **`Dpar` and `Dperp`.** Only `Dpar / Dperp` matters. A ratio of 1 is the isotropic flow, and the larger the ratio the stronger the steering. `Dperp` can be 0 to stop the flow across the direction altogether, but then regions can hardly change in that direction and the morph may not converge.
- **Convergence.** Check the status and the error after every run, as above. A strong ratio, or a direction that conflicts with the data (a region that has to grow across the preferred direction), converges more slowly; raise `n_iter` or lower the ratio.
- **Other directions.** `theta` can also be a function of the grid, an array, or `DirectionalTensor.from_seeds` for directions set at a few places; see [Controlling Geographical Distortions](reduce-boundary-distortion.ipynb).
- **Combining.** Modulators can be added into a pipeline, for example `DirectionalTensor(0.0, Dpar=3.0) + VelocitySmooth(5)`.